In [1]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F

In [2]:
spark = SparkSession.builder.getOrCreate()

spark.sparkContext

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
/usr/local/lib/python3.12/site-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
26/11/12 12:31:55 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


<SparkContext master=local[*] appName=pyspark-shell>

# **Tabular Data**

In [3]:
my_grocery_list = [
    ["Banana", 2, 1.74],
    ["Apple", 4, 2.04],
    ["Carrot", 1, 1.09],
    ["Cake", 1, 10.99],
]

df_grocery_list = spark.createDataFrame(
    my_grocery_list, ["Item", "Quantity", "Price"]
)

df_grocery_list.printSchema()

root
 |-- Item: string (nullable = true)
 |-- Quantity: long (nullable = true)
 |-- Price: double (nullable = true)



![](../images/Cluster_Memory.png)

In [4]:
import os

In [5]:
DIRECTORY = "../data/broadcast_logs"
df_logs = spark.read.csv(
    os.path.join(DIRECTORY, 'BroadcastLogs_2018_Q3_M8_sample.CSV'),
    sep='|',
    inferSchema=True,
    header=True,
    timestampFormat="yyyy-MM-dd"
)

df_logs.printSchema()

root
 |-- BroadcastLogID: integer (nullable = true)
 |-- LogServiceID: integer (nullable = true)
 |-- LogDate: date (nullable = true)
 |-- SequenceNO: integer (nullable = true)
 |-- AudienceTargetAgeID: integer (nullable = true)
 |-- AudienceTargetEthnicID: integer (nullable = true)
 |-- CategoryID: integer (nullable = true)
 |-- ClosedCaptionID: integer (nullable = true)
 |-- CountryOfOriginID: integer (nullable = true)
 |-- DubDramaCreditID: integer (nullable = true)
 |-- EthnicProgramID: integer (nullable = true)
 |-- ProductionSourceID: integer (nullable = true)
 |-- ProgramClassID: integer (nullable = true)
 |-- FilmClassificationID: integer (nullable = true)
 |-- ExhibitionID: integer (nullable = true)
 |-- Duration: string (nullable = true)
 |-- EndTime: string (nullable = true)
 |-- LogEntryDate: date (nullable = true)
 |-- ProductionNO: string (nullable = true)
 |-- ProgramTitle: string (nullable = true)
 |-- StartTime: string (nullable = true)
 |-- Subtitle: string (nullable 

# **Data Manipulation**

## **Selecting columns**

In [6]:
df_logs.select("BroadcastLogID", "LogServiceID", "LogDate").show(5, False)

+--------------+------------+----------+
|BroadcastLogID|LogServiceID|LogDate   |
+--------------+------------+----------+
|1196192316    |3157        |2018-08-01|
|1196192317    |3157        |2018-08-01|
|1196192318    |3157        |2018-08-01|
|1196192319    |3157        |2018-08-01|
|1196192320    |3157        |2018-08-01|
+--------------+------------+----------+
only showing top 5 rows


In [7]:
df_logs.select(*["BroadcastLogID", "LogServiceID", "LogDate"]).show(5, False)

+--------------+------------+----------+
|BroadcastLogID|LogServiceID|LogDate   |
+--------------+------------+----------+
|1196192316    |3157        |2018-08-01|
|1196192317    |3157        |2018-08-01|
|1196192318    |3157        |2018-08-01|
|1196192319    |3157        |2018-08-01|
|1196192320    |3157        |2018-08-01|
+--------------+------------+----------+
only showing top 5 rows


In [8]:
import numpy as np

In [9]:
splited_columns = np.array_split(
    df_logs.columns, len(df_logs.columns) // 3
)


for x in splited_columns:
    df_logs.select(*x).show(5, truncate=False)

+--------------+------------+----------+
|BroadcastLogID|LogServiceID|LogDate   |
+--------------+------------+----------+
|1196192316    |3157        |2018-08-01|
|1196192317    |3157        |2018-08-01|
|1196192318    |3157        |2018-08-01|
|1196192319    |3157        |2018-08-01|
|1196192320    |3157        |2018-08-01|
+--------------+------------+----------+
only showing top 5 rows
+----------+-------------------+----------------------+
|SequenceNO|AudienceTargetAgeID|AudienceTargetEthnicID|
+----------+-------------------+----------------------+
|1         |4                  |NULL                  |
|2         |NULL               |NULL                  |
|3         |NULL               |NULL                  |
|4         |NULL               |NULL                  |
|5         |NULL               |NULL                  |
+----------+-------------------+----------------------+
only showing top 5 rows
+----------+---------------+-----------------+
|CategoryID|ClosedCaptionID|Coun

## **Deleting columns**

In [10]:
df_logs = df_logs.drop("BroadcastLogID", "SequenceNO")

print("BroadcastLogID" in df_logs.columns) # => False
print("SequenceNo" in df_logs.columns) # => False

False
False


**Advanced topic: An unfortunate inconsistency**

In theory, you can also select() columns with a list without unpacking them. This code will work as expected:
```spark
logs = logs.select(
    [x for x in logs.columns if x not in ["BroadcastLogID", "SequenceNO"]]
)
```
This is not the case for drop(), where you need to explicitly unpack:
```spark
logs.drop(logs.columns[:])
# TypeError: col should be a string or a Column
logs.drop(*logs.columns[:])
# DataFrame[]
```

## **New columns with `withColumn()`**

In [11]:
df_logs.select(F.col("Duration")).show(5)

+----------------+
|        Duration|
+----------------+
|02:00:00.0000000|
|00:00:30.0000000|
|00:00:15.0000000|
|00:00:15.0000000|
|00:00:15.0000000|
+----------------+
only showing top 5 rows


In [12]:
df_logs.select(F.col('Duration')).dtypes

[('Duration', 'string')]

In [13]:
df_logs.select(
    F.col("Duration"),
    F.col("Duration").substr(1, 2).cast("int").alias("dur_hours"),
    F.col("Duration").substr(4, 2).cast("int").alias("dur_minutes"),
    F.col("Duration").substr(7, 2).cast("int").alias("dur_seconds"),
).distinct().show(5, False)

+----------------+---------+-----------+-----------+
|Duration        |dur_hours|dur_minutes|dur_seconds|
+----------------+---------+-----------+-----------+
|00:04:52.0000000|0        |4          |52         |
|00:10:06.0000000|0        |10         |6          |
|00:26:41.0000000|0        |26         |41         |
|00:05:29.0000000|0        |5          |29         |
|00:08:18.0000000|0        |8          |18         |
+----------------+---------+-----------+-----------+
only showing top 5 rows


In [14]:
df_logs.select(
    F.col("Duration"),
    (
        F.col("Duration").substr(1, 2).cast('int') * 120 
        + F.col("Duration").substr(4, 2).cast('int') * 60 
        + F.col("Duration").substr(7, 2).cast('int') 
    ).alias("Duration in second")
).distinct().show(5, False)

+----------------+------------------+
|Duration        |Duration in second|
+----------------+------------------+
|00:31:00.0000000|1860              |
|00:28:08.0000000|1688              |
|00:10:30.0000000|630               |
|00:32:00.0000000|1920              |
|00:30:00.0000000|1800              |
+----------------+------------------+
only showing top 5 rows


In [15]:
df_logs = df_logs.withColumn(
    "Duration_Seconds",
    (
        F.col("Duration").substr(1, 2).cast('int') * 120 
        + F.col("Duration").substr(4, 2).cast('int') * 60 
        + F.col("Duration").substr(7, 2).cast('int') 
    )
)

df_logs.printSchema()

root
 |-- LogServiceID: integer (nullable = true)
 |-- LogDate: date (nullable = true)
 |-- AudienceTargetAgeID: integer (nullable = true)
 |-- AudienceTargetEthnicID: integer (nullable = true)
 |-- CategoryID: integer (nullable = true)
 |-- ClosedCaptionID: integer (nullable = true)
 |-- CountryOfOriginID: integer (nullable = true)
 |-- DubDramaCreditID: integer (nullable = true)
 |-- EthnicProgramID: integer (nullable = true)
 |-- ProductionSourceID: integer (nullable = true)
 |-- ProgramClassID: integer (nullable = true)
 |-- FilmClassificationID: integer (nullable = true)
 |-- ExhibitionID: integer (nullable = true)
 |-- Duration: string (nullable = true)
 |-- EndTime: string (nullable = true)
 |-- LogEntryDate: date (nullable = true)
 |-- ProductionNO: string (nullable = true)
 |-- ProgramTitle: string (nullable = true)
 |-- StartTime: string (nullable = true)
 |-- Subtitle: string (nullable = true)
 |-- NetworkAffiliationID: integer (nullable = true)
 |-- SpecialAttentionID: inte

> **WARNING :** If you create a column withColumn() and give it a name that already exists in your data frame, PySpark will happily overwrite the column.

> **WARNING :** Creating many (100+) new columns using `withColumns()` will slow Spark down to a grind. If you need to create a lot of columns at once, use the `select()` approach. While it will generate the same work, it is less tasking on the query planner.

![](../images/select_versus_withColumn.png)

## **Renaming and reordering columns**

In [16]:
df_logs = df_logs.withColumnRenamed(
    'Duration_Seconds', 'duration_seconds'
)

df_logs.printSchema()

root
 |-- LogServiceID: integer (nullable = true)
 |-- LogDate: date (nullable = true)
 |-- AudienceTargetAgeID: integer (nullable = true)
 |-- AudienceTargetEthnicID: integer (nullable = true)
 |-- CategoryID: integer (nullable = true)
 |-- ClosedCaptionID: integer (nullable = true)
 |-- CountryOfOriginID: integer (nullable = true)
 |-- DubDramaCreditID: integer (nullable = true)
 |-- EthnicProgramID: integer (nullable = true)
 |-- ProductionSourceID: integer (nullable = true)
 |-- ProgramClassID: integer (nullable = true)
 |-- FilmClassificationID: integer (nullable = true)
 |-- ExhibitionID: integer (nullable = true)
 |-- Duration: string (nullable = true)
 |-- EndTime: string (nullable = true)
 |-- LogEntryDate: date (nullable = true)
 |-- ProductionNO: string (nullable = true)
 |-- ProgramTitle: string (nullable = true)
 |-- StartTime: string (nullable = true)
 |-- Subtitle: string (nullable = true)
 |-- NetworkAffiliationID: integer (nullable = true)
 |-- SpecialAttentionID: inte

In [17]:
df_logs.toDF(*[x.lower() for x in df_logs.columns]).printSchema()

root
 |-- logserviceid: integer (nullable = true)
 |-- logdate: date (nullable = true)
 |-- audiencetargetageid: integer (nullable = true)
 |-- audiencetargetethnicid: integer (nullable = true)
 |-- categoryid: integer (nullable = true)
 |-- closedcaptionid: integer (nullable = true)
 |-- countryoforiginid: integer (nullable = true)
 |-- dubdramacreditid: integer (nullable = true)
 |-- ethnicprogramid: integer (nullable = true)
 |-- productionsourceid: integer (nullable = true)
 |-- programclassid: integer (nullable = true)
 |-- filmclassificationid: integer (nullable = true)
 |-- exhibitionid: integer (nullable = true)
 |-- duration: string (nullable = true)
 |-- endtime: string (nullable = true)
 |-- logentrydate: date (nullable = true)
 |-- productionno: string (nullable = true)
 |-- programtitle: string (nullable = true)
 |-- starttime: string (nullable = true)
 |-- subtitle: string (nullable = true)
 |-- networkaffiliationid: integer (nullable = true)
 |-- specialattentionid: inte

In [18]:
df_logs.select(sorted(df_logs.columns)).printSchema()

root
 |-- AudienceTargetAgeID: integer (nullable = true)
 |-- AudienceTargetEthnicID: integer (nullable = true)
 |-- BroadcastOriginPointID: integer (nullable = true)
 |-- CategoryID: integer (nullable = true)
 |-- ClosedCaptionID: integer (nullable = true)
 |-- CompositionID: integer (nullable = true)
 |-- CountryOfOriginID: integer (nullable = true)
 |-- DubDramaCreditID: integer (nullable = true)
 |-- Duration: string (nullable = true)
 |-- EndTime: string (nullable = true)
 |-- EthnicProgramID: integer (nullable = true)
 |-- ExhibitionID: integer (nullable = true)
 |-- FilmClassificationID: integer (nullable = true)
 |-- Language1: integer (nullable = true)
 |-- Language2: integer (nullable = true)
 |-- LogDate: date (nullable = true)
 |-- LogEntryDate: date (nullable = true)
 |-- LogServiceID: integer (nullable = true)
 |-- NetworkAffiliationID: integer (nullable = true)
 |-- Producer1: string (nullable = true)
 |-- Producer2: string (nullable = true)
 |-- ProductionNO: string (nu

## **Diagnosing a Data Frame**

In [19]:
for x in df_logs.columns:
    print(df_logs[x].dtype)
    df_logs.describe(x).show()

Column<'LogServiceID['dtype']'>
+-------+------------------+
|summary|      LogServiceID|
+-------+------------------+
|  count|            238945|
|   mean| 3450.890284375065|
| stddev|199.50673962555987|
|    min|              3157|
|    max|              3925|
+-------+------------------+

Column<'LogDate['dtype']'>
+-------+
|summary|
+-------+
|  count|
|   mean|
| stddev|
|    min|
|    max|
+-------+

Column<'AudienceTargetAgeID['dtype']'>
+-------+-------------------+
|summary|AudienceTargetAgeID|
+-------+-------------------+
|  count|              16112|
|   mean| 3.4929245283018866|
| stddev|  1.041596339474513|
|    min|                  1|
|    max|                  4|
+-------+-------------------+

Column<'AudienceTargetEthnicID['dtype']'>
+-------+----------------------+
|summary|AudienceTargetEthnicID|
+-------+----------------------+
|  count|                  1710|
|   mean|    120.56432748538012|
| stddev|     71.98694059436134|
|    min|                     4|
|    

In [20]:
for x in df_logs.columns:
    df_logs.select(x).summary("min", "10%", "90%", "max").show()

+-------+------------+
|summary|LogServiceID|
+-------+------------+
|    min|        3157|
|    10%|        3236|
|    90%|        3709|
|    max|        3925|
+-------+------------+

+-------+
|summary|
+-------+
|    min|
|    10%|
|    90%|
|    max|
+-------+

+-------+-------------------+
|summary|AudienceTargetAgeID|
+-------+-------------------+
|    min|                  1|
|    10%|                  1|
|    90%|                  4|
|    max|                  4|
+-------+-------------------+

+-------+----------------------+
|summary|AudienceTargetEthnicID|
+-------+----------------------+
|    min|                     4|
|    10%|                    74|
|    90%|                   258|
|    max|                   337|
+-------+----------------------+

+-------+----------+
|summary|CategoryID|
+-------+----------+
|    min|         1|
|    10%|         3|
|    90%|        29|
|    max|        29|
+-------+----------+

+-------+---------------+
|summary|ClosedCaptionID|
+------

+-------+--------------+
|summary|ProgramClassID|
+-------+--------------+
|    min|             3|
|    10%|             3|
|    90%|            20|
|    max|            30|
+-------+--------------+

+-------+--------------------+
|summary|FilmClassificationID|
+-------+--------------------+
|    min|                   1|
|    10%|                   1|
|    90%|                   5|
|    max|                   8|
+-------+--------------------+

+-------+------------+
|summary|ExhibitionID|
+-------+------------+
|    min|           1|
|    10%|           2|
|    90%|           5|
|    max|           8|
+-------+------------+

+-------+----------------+
|summary|        Duration|
+-------+----------------+
|    min|00:00:01.0000000|
|    10%|            NULL|
|    90%|            NULL|
|    max|06:30:09.0000000|
+-------+----------------+



+-------+----------------+
|summary|         EndTime|
+-------+----------------+
|    min|00:00:00.0000000|
|    10%|            NULL|
|    90%|            NULL|
|    max|23:59:59.0000000|
+-------+----------------+

+-------+
|summary|
+-------+
|    min|
|    10%|
|    90%|
|    max|
+-------+

+-------+------------+
|summary|ProductionNO|
+-------+------------+
|    min|      030641|
|    10%|     30641.0|
|    90%|     39949.0|
|    max|      c34183|
+-------+------------+

+-------+------------------+
|summary|      ProgramTitle|
+-------+------------------+
|    min|  !NO 5PM A ID(:5)|
|    10%|            1999.0|
|    90%|            1999.0|
|    max|�t� avec Jo�l 2/Un|
+-------+------------------+

+-------+----------------+
|summary|       StartTime|
+-------+----------------+
|    min|00:00:00.0000000|
|    10%|            NULL|
|    90%|            NULL|
|    max|23:59:59.0000000|
+-------+----------------+

+-------+--------------------+
|summary|            Subtitle|
+----

# **Additional exercises**

#### **Exercise 4.3**
Reread the data in a logs_raw data frame (the data file is `./data/broadcast_logs-BroadcastLogs_2018_Q3_M8.CSV)`, this time without passing any optional parameters Print the first five rows of data, as well as the schema. What are the differences in terms of data and schema between logs and logs_raw?

In [21]:
df_logs_raw = spark.read.csv('../data/broadcast_logs/BroadcastLogs_2018_Q3_M8_sample.CSV')

df_logs_raw.printSchema()

root
 |-- _c0: string (nullable = true)



In [22]:
df_logs_raw.show(5, False)

+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|_c0                                                                                                                                                                                                                                                                                                                                                                                                                                        |
+-------------------------------------------------------------------------------------------------------------------------------------------

#### **Exercise 4.4**
Create a new data frame, logs_clean, that contains only the columns that do not end with ID.

In [23]:
DIRECTORY

'../data/broadcast_logs'

In [25]:
df_clean_logs = spark.read.csv(
    os.path.join(DIRECTORY, 'BroadcastLogs_2018_Q3_M8_sample.CSV'),
    sep= '|',
    inferSchema=True,
    header=True,
    timestampFormat='yyyy/MM/DD'
)

df_clean_logs.printSchema()

root
 |-- BroadcastLogID: integer (nullable = true)
 |-- LogServiceID: integer (nullable = true)
 |-- LogDate: date (nullable = true)
 |-- SequenceNO: integer (nullable = true)
 |-- AudienceTargetAgeID: integer (nullable = true)
 |-- AudienceTargetEthnicID: integer (nullable = true)
 |-- CategoryID: integer (nullable = true)
 |-- ClosedCaptionID: integer (nullable = true)
 |-- CountryOfOriginID: integer (nullable = true)
 |-- DubDramaCreditID: integer (nullable = true)
 |-- EthnicProgramID: integer (nullable = true)
 |-- ProductionSourceID: integer (nullable = true)
 |-- ProgramClassID: integer (nullable = true)
 |-- FilmClassificationID: integer (nullable = true)
 |-- ExhibitionID: integer (nullable = true)
 |-- Duration: string (nullable = true)
 |-- EndTime: string (nullable = true)
 |-- LogEntryDate: date (nullable = true)
 |-- ProductionNO: string (nullable = true)
 |-- ProgramTitle: string (nullable = true)
 |-- StartTime: string (nullable = true)
 |-- Subtitle: string (nullable 

In [26]:
selected_cols = [x for x in df_clean_logs.columns if x.endswith("ID")]

df_clean_logs.drop(*selected_cols).printSchema()

root
 |-- LogDate: date (nullable = true)
 |-- SequenceNO: integer (nullable = true)
 |-- Duration: string (nullable = true)
 |-- EndTime: string (nullable = true)
 |-- LogEntryDate: date (nullable = true)
 |-- ProductionNO: string (nullable = true)
 |-- ProgramTitle: string (nullable = true)
 |-- StartTime: string (nullable = true)
 |-- Subtitle: string (nullable = true)
 |-- Producer1: string (nullable = true)
 |-- Producer2: string (nullable = true)
 |-- Language1: integer (nullable = true)
 |-- Language2: integer (nullable = true)



# **Joining and grouping**

## **Joining data**

In [34]:
df_logs = spark.read.csv(
    os.path.join(DIRECTORY, 'BroadcastLogs_2018_Q3_M8_sample.CSV'),
    sep='|',
    header=True,
    inferSchema=True,
    timestampFormat='yyyy-MM-DD'
)


df_log_identifier = spark.read.csv(
    os.path.join(DIRECTORY, 'ReferenceTables/LogIdentifier.csv'),
    sep='|',
    header=True,
    inferSchema=True
)


df_logs.printSchema()
df_log_identifier.printSchema()

root
 |-- BroadcastLogID: integer (nullable = true)
 |-- LogServiceID: integer (nullable = true)
 |-- LogDate: date (nullable = true)
 |-- SequenceNO: integer (nullable = true)
 |-- AudienceTargetAgeID: integer (nullable = true)
 |-- AudienceTargetEthnicID: integer (nullable = true)
 |-- CategoryID: integer (nullable = true)
 |-- ClosedCaptionID: integer (nullable = true)
 |-- CountryOfOriginID: integer (nullable = true)
 |-- DubDramaCreditID: integer (nullable = true)
 |-- EthnicProgramID: integer (nullable = true)
 |-- ProductionSourceID: integer (nullable = true)
 |-- ProgramClassID: integer (nullable = true)
 |-- FilmClassificationID: integer (nullable = true)
 |-- ExhibitionID: integer (nullable = true)
 |-- Duration: string (nullable = true)
 |-- EndTime: string (nullable = true)
 |-- LogEntryDate: date (nullable = true)
 |-- ProductionNO: string (nullable = true)
 |-- ProgramTitle: string (nullable = true)
 |-- StartTime: string (nullable = true)
 |-- Subtitle: string (nullable 

In [35]:
df_log_identifier.show(5, False)

+---------------+------------+---------+
|LogIdentifierID|LogServiceID|PrimaryFG|
+---------------+------------+---------+
|13ST           |3157        |1        |
|2000SM         |3466        |1        |
|70SM           |3883        |1        |
|80SM           |3590        |1        |
|90SM           |3470        |1        |
+---------------+------------+---------+
only showing top 5 rows


In [45]:
df_log_identifier = df_log_identifier.where(df_log_identifier['PrimaryFG'] == 1)

df_log_identifier.count()

758

### **The join method**

![](../images/inner_join.png)
![](../images/left_rigth_join.png)
![](../images/outer_join.png)

In [46]:
logs_and_channels = df_logs.join(
    df_log_identifier,
    on='LogServiceID',
    how='inner'
)

In Apache Spark and SQL, **Left Semi-Join** and **Left Anti-Join** are specialized filtering joins. Unlike standard Inner or Left Joins, they **never add columns from the right DataFrame** to the output. Instead, they use the right DataFrame purely as a filter condition against the left DataFrame.

---

#### **1. Left Semi-Join (`"leftsemi"`)**

A **Left Semi-Join** returns all rows from the **left DataFrame** where there is a matching record in the **right DataFrame**.

* **SQL Equivalent:** Similar to an `IN` or `EXISTS` clause.
* **Key Behavior:**
* Only columns from the **left** DataFrame are kept.
* If a row in the left table has *multiple* matches in the right table, a standard inner join would duplicate that row. A **semi-join guarantees each row from the left table is returned at most once** (no duplication).



#### **PySpark Code Example:**

```python
# Employees DataFrame
employees = spark.createDataFrame(
    [
        (1, "Alice", "HR"),
        (2, "Bob", "Engineering"),
        (3, "Charlie", "Marketing"),
        (4, "David", "Sales"),
    ],
    ["id", "name", "department"],
)

# Active department IDs list/DataFrame
active_depts = spark.createDataFrame(
    [("HR",), ("Engineering",)], ["department"]
)

# Left Semi-Join: Get employees who work in an active department
# Notice: Only columns from the 'employees' DataFrame are returned.
result_semi = employees.join(
    active_depts,
    employees.department == active_depts.department,
    "leftsemi",  # Specify the join type
)

result_semi.show()
# Output will contain Alice (HR) and Bob (Engineering), but NO columns from active_depts.

```

---

#### **2. Left Anti-Join (`"leftanti"`)**

A **Left Anti-Join** is the exact opposite of a left semi-join. It returns all rows from the **left DataFrame** where there is **NO matching record** in the **right DataFrame**.

* **SQL Equivalent:** Similar to a `NOT IN` or `NOT EXISTS` clause.
* **Key Behavior:**
* Only columns from the **left** DataFrame are kept.
* It acts as an exclusion filter, perfect for finding "orphan" records, missing data, or items that haven't been processed yet.



#### **PySpark Code Example:**

```python
# Left Anti-Join: Get employees who do NOT work in active departments (e.g., Marketing, Sales)
result_anti = employees.join(
    active_depts,
    employees.department == active_depts.department,
    "leftanti",  # Specify the anti-join type
)

result_anti.show()
# Output will contain Charlie (Marketing) and David (Sales) because they have no match in active_depts.

```

---

#### **📊 Summary Comparison**

| Join Type | What it Keeps | Column Output | Duplication Risk? | Common Use Case |
| --- | --- | --- | --- | --- |
| **Left Semi** (`"leftsemi"`) | Rows from Left **that HAVE a match** in Right | Left columns only | No (returns unique left rows) | Filtering datasets based on existence in another table (`IN`). |
| **Left Anti** (`"leftanti"`) | Rows from Left **that DO NOT have a match** in Right | Left columns only | No | Finding missing records, errors, or unprocessed items (`NOT IN`). |

![](../images/cross_join.png)

In a distributed computing environment like Apache Spark, joining operations are notoriously the most common performance bottlenecks (often referred to as the "weakness" or heavy lifting of distributed processing).

Unlike a single-machine database where tables live on the same local disk, a Spark cluster splits data across multiple worker nodes. For two tables to be joined, **rows with matching join keys must meet on the exact same physical worker node**.

Here is why joining operations present such a major challenge under the hood:

---

#### **1. The Network Shuffle (The Heavy Cost of Moving Data)**

If your datasets are not already partitioned by the join key, Spark cannot perform the join locally.

* It must initiate a **Shuffle**: a wide transformation where data is read across the network, re-hashed, and redistributed so that matching keys land on the same executor.
* **Why it hurts:** Network I/O is orders of magnitude slower than reading from RAM or local disk. Shuffling massive datasets creates heavy network congestion and serialization/deserialization overhead.

#### **2. Data Skew (The "Straggler" Problem)**

In real-world data, keys are rarely distributed evenly. For example, if you join user logs with a user table, a few "null" keys or popular bot/system accounts might account for 50% of the rows.

* **Why it hurts:** Spark assigns one task per partition. If one partition contains 90% of the matching join keys due to skew, the worker handling that partition will be completely overwhelmed with data. It will take ten times longer to finish than all other workers, turning into a **straggler task** that stalls the entire job and frequently leads to `OutOftMemory (OOM)` errors.

#### **3. Memory Pressure and Disk Spill**

To execute standard joins (like a Sort-Merge Join), Spark often needs to sort and buffer large chunks of data in memory on the worker nodes.

* **Why it hurts:** If the data partitions being joined exceed available executor memory, Spark is forced to **spill data to disk**. Writing and reading intermediate join blocks from local disk dramatically degrades query performance.

#### **4. Combinatorial Explosion (Cartesian Product Risk)**

If a join condition is written incorrectly, poorly thought out, or accidentally turns into an unconstrained cross-join (matching every row of table A to every row of table B), the output size doesn't just add up—it multiplies ($N \times M$).

* **Why it hurts:** A single flawed join can generate billions or trillions of rows, instantly flooding the cluster's memory and crashing the application.

---

#### **💡 How Spark Engineers Mitigate Join Weaknesses**

Because joins are so heavy, Spark and data engineers use specific optimization strategies:

* **Broadcast Hash Joins (Broadcast Joins):** If one table is small, Spark avoids the shuffle entirely by copying (broadcasting) the small table to every worker node, allowing each node to perform the join locally in memory.
* **Co-partitioning:** Pre-partitioning both datasets by the join key prior to writing them to storage ensures they are already aligned, skipping the shuffle phase entirely during the join.

In [48]:
logs_and_channels_verbose = df_logs.join(
    df_log_identifier,
    df_logs['LogServiceID'] == df_log_identifier['LogServiceID']
)

logs_and_channels.printSchema()
logs_and_channels_verbose.printSchema()

root
 |-- LogServiceID: integer (nullable = true)
 |-- BroadcastLogID: integer (nullable = true)
 |-- LogDate: date (nullable = true)
 |-- SequenceNO: integer (nullable = true)
 |-- AudienceTargetAgeID: integer (nullable = true)
 |-- AudienceTargetEthnicID: integer (nullable = true)
 |-- CategoryID: integer (nullable = true)
 |-- ClosedCaptionID: integer (nullable = true)
 |-- CountryOfOriginID: integer (nullable = true)
 |-- DubDramaCreditID: integer (nullable = true)
 |-- EthnicProgramID: integer (nullable = true)
 |-- ProductionSourceID: integer (nullable = true)
 |-- ProgramClassID: integer (nullable = true)
 |-- FilmClassificationID: integer (nullable = true)
 |-- ExhibitionID: integer (nullable = true)
 |-- Duration: string (nullable = true)
 |-- EndTime: string (nullable = true)
 |-- LogEntryDate: date (nullable = true)
 |-- ProductionNO: string (nullable = true)
 |-- ProgramTitle: string (nullable = true)
 |-- StartTime: string (nullable = true)
 |-- Subtitle: string (nullable 

In [49]:
from pyspark.errors import AnalysisException

In [50]:
try:
    logs_and_channels_verbose.select('LogServiceID')
except AnalysisException as err:
    print(err)

{"ts": "2026-11-12 13:19:33.742", "level": "ERROR", "logger": "DataFrameQueryContextLogger", "msg": "[AMBIGUOUS_REFERENCE] Reference `LogServiceID` is ambiguous, could be: [`LogServiceID`, `LogServiceID`]. SQLSTATE: 42704", "context": {"file": "java.base/jdk.internal.reflect.DirectMethodHandleAccessor.invoke(DirectMethodHandleAccessor.java", "line": "103)", "fragment": "col", "errorClass": "AMBIGUOUS_REFERENCE"}, "exception": {"class": "Py4JJavaError", "msg": "An error occurred while calling o1184.select.\n: org.apache.spark.sql.AnalysisException: [AMBIGUOUS_REFERENCE] Reference `LogServiceID` is ambiguous, could be: [`LogServiceID`, `LogServiceID`]. SQLSTATE: 42704\n\tat org.apache.spark.sql.errors.QueryCompilationErrors$.ambiguousReferenceError(QueryCompilationErrors.scala:2326)\n\tat org.apache.spark.sql.catalyst.expressions.package$AttributeSeq.resolve(package.scala:363)\n\tat org.apache.spark.sql.catalyst.plans.logical.LogicalPlan.resolveChildren(LogicalPlan.scala:164)\n\tat org.a

[AMBIGUOUS_REFERENCE] Reference `LogServiceID` is ambiguous, could be: [`LogServiceID`, `LogServiceID`]. SQLSTATE: 42704


In [51]:
logs_and_channels = df_logs.join(
    df_log_identifier,
    'LogServiceID'
)

logs_and_channels.printSchema()

root
 |-- LogServiceID: integer (nullable = true)
 |-- BroadcastLogID: integer (nullable = true)
 |-- LogDate: date (nullable = true)
 |-- SequenceNO: integer (nullable = true)
 |-- AudienceTargetAgeID: integer (nullable = true)
 |-- AudienceTargetEthnicID: integer (nullable = true)
 |-- CategoryID: integer (nullable = true)
 |-- ClosedCaptionID: integer (nullable = true)
 |-- CountryOfOriginID: integer (nullable = true)
 |-- DubDramaCreditID: integer (nullable = true)
 |-- EthnicProgramID: integer (nullable = true)
 |-- ProductionSourceID: integer (nullable = true)
 |-- ProgramClassID: integer (nullable = true)
 |-- FilmClassificationID: integer (nullable = true)
 |-- ExhibitionID: integer (nullable = true)
 |-- Duration: string (nullable = true)
 |-- EndTime: string (nullable = true)
 |-- LogEntryDate: date (nullable = true)
 |-- ProductionNO: string (nullable = true)
 |-- ProgramTitle: string (nullable = true)
 |-- StartTime: string (nullable = true)
 |-- Subtitle: string (nullable 

In [52]:
logs_and_channels = df_logs.join(
    df_log_identifier,
    df_logs['LogServiceID'] == df_log_identifier['LogServiceID']
)

logs_and_channels.drop(df_log_identifier['LogServiceID']).select('LogServiceID')

DataFrame[LogServiceID: int]

In [54]:
logs_and_channels = df_logs.alias('left').join(
    df_log_identifier.alias('rigth'),
    df_logs['LogServiceID'] == df_log_identifier['LogServiceID']
)

logs_and_channels.drop(F.col('rigth.LogServiceID')).select('LogServiceID')

DataFrame[LogServiceID: int]

In [55]:
logs_and_channels = df_logs.join(
    df_log_identifier,
    on='LogServiceID',
    how='inner'
)

logs_and_channels.printSchema()

root
 |-- LogServiceID: integer (nullable = true)
 |-- BroadcastLogID: integer (nullable = true)
 |-- LogDate: date (nullable = true)
 |-- SequenceNO: integer (nullable = true)
 |-- AudienceTargetAgeID: integer (nullable = true)
 |-- AudienceTargetEthnicID: integer (nullable = true)
 |-- CategoryID: integer (nullable = true)
 |-- ClosedCaptionID: integer (nullable = true)
 |-- CountryOfOriginID: integer (nullable = true)
 |-- DubDramaCreditID: integer (nullable = true)
 |-- EthnicProgramID: integer (nullable = true)
 |-- ProductionSourceID: integer (nullable = true)
 |-- ProgramClassID: integer (nullable = true)
 |-- FilmClassificationID: integer (nullable = true)
 |-- ExhibitionID: integer (nullable = true)
 |-- Duration: string (nullable = true)
 |-- EndTime: string (nullable = true)
 |-- LogEntryDate: date (nullable = true)
 |-- ProductionNO: string (nullable = true)
 |-- ProgramTitle: string (nullable = true)
 |-- StartTime: string (nullable = true)
 |-- Subtitle: string (nullable 

In [56]:
cd_category = spark.read.csv(
    os.path.join(DIRECTORY, "ReferenceTables/CD_Category.csv"),
    sep="|",
    header=True,
    inferSchema=True,
    ).select(
    "CategoryID",
    "CategoryCD",
    F.col("EnglishDescription").alias("Category_Description"),
)

cd_program_class = spark.read.csv(
    os.path.join(DIRECTORY, "ReferenceTables/CD_ProgramClass.csv"),
    sep="|",
    header=True,
    inferSchema=True,
    ).select(
    "ProgramClassID",
    "ProgramClassCD",
    F.col("EnglishDescription").alias("ProgramClass_Description"),
)

full_log = logs_and_channels.join(
    cd_category, 
    "CategoryID", 
    how="left").join(
    cd_program_class, 
    "ProgramClassID", 
    how="left"
)

full_log.printSchema()

root
 |-- ProgramClassID: integer (nullable = true)
 |-- CategoryID: integer (nullable = true)
 |-- LogServiceID: integer (nullable = true)
 |-- BroadcastLogID: integer (nullable = true)
 |-- LogDate: date (nullable = true)
 |-- SequenceNO: integer (nullable = true)
 |-- AudienceTargetAgeID: integer (nullable = true)
 |-- AudienceTargetEthnicID: integer (nullable = true)
 |-- ClosedCaptionID: integer (nullable = true)
 |-- CountryOfOriginID: integer (nullable = true)
 |-- DubDramaCreditID: integer (nullable = true)
 |-- EthnicProgramID: integer (nullable = true)
 |-- ProductionSourceID: integer (nullable = true)
 |-- FilmClassificationID: integer (nullable = true)
 |-- ExhibitionID: integer (nullable = true)
 |-- Duration: string (nullable = true)
 |-- EndTime: string (nullable = true)
 |-- LogEntryDate: date (nullable = true)
 |-- ProductionNO: string (nullable = true)
 |-- ProgramTitle: string (nullable = true)
 |-- StartTime: string (nullable = true)
 |-- Subtitle: string (nullable 

### **Grouping Data**

In [64]:
full_log.select(['Duration']).show(5, False)

+----------------+
|Duration        |
+----------------+
|02:00:00.0000000|
|00:00:30.0000000|
|00:00:15.0000000|
|00:00:15.0000000|
|00:00:15.0000000|
+----------------+
only showing top 5 rows


In [73]:
full_log = full_log.withColumn(
    'duration_seconds',
    (F.col('Duration').substr(1, 2).cast('int') * 3600
    + F.col('Duration').substr(4, 2).cast('int') * 60
    + F.col('Duration').substr(7, 2).cast('int'))
)

In [74]:
full_log.printSchema()

root
 |-- ProgramClassID: integer (nullable = true)
 |-- CategoryID: integer (nullable = true)
 |-- LogServiceID: integer (nullable = true)
 |-- BroadcastLogID: integer (nullable = true)
 |-- LogDate: date (nullable = true)
 |-- SequenceNO: integer (nullable = true)
 |-- AudienceTargetAgeID: integer (nullable = true)
 |-- AudienceTargetEthnicID: integer (nullable = true)
 |-- ClosedCaptionID: integer (nullable = true)
 |-- CountryOfOriginID: integer (nullable = true)
 |-- DubDramaCreditID: integer (nullable = true)
 |-- EthnicProgramID: integer (nullable = true)
 |-- ProductionSourceID: integer (nullable = true)
 |-- FilmClassificationID: integer (nullable = true)
 |-- ExhibitionID: integer (nullable = true)
 |-- Duration: string (nullable = true)
 |-- EndTime: string (nullable = true)
 |-- LogEntryDate: date (nullable = true)
 |-- ProductionNO: string (nullable = true)
 |-- ProgramTitle: string (nullable = true)
 |-- StartTime: string (nullable = true)
 |-- Subtitle: string (nullable 

In [76]:
(
    full_log
    .groupby("ProgramClassCD", "ProgramClass_Description")
    .agg(F.sum("duration_seconds").alias("duration_total"))
    .orderBy("duration_total", ascending=False).show(100, False)
)

+--------------+--------------------------------------+--------------+
|ProgramClassCD|ProgramClass_Description              |duration_total|
+--------------+--------------------------------------+--------------+
|PGR           |PROGRAM                               |20992510      |
|COM           |COMMERCIAL MESSAGE                    |3519163       |
|PFS           |PROGRAM FIRST SEGMENT                 |1344762       |
|SEG           |SEGMENT OF A PROGRAM                  |1205998       |
|PRC           |PROMOTION OF UPCOMING CANADIAN PROGRAM|880600        |
|PGI           |PROGRAM INFOMERCIAL                   |679182        |
|PRO           |PROMOTION OF NON-CANADIAN PROGRAM     |335701        |
|OFF           |SCHEDULED OFF AIR TIME PERIOD         |142279        |
|ID            |NETWORK IDENTIFICATION MESSAGE        |74926         |
|NRN           |No recognized nationality             |59686         |
|MAG           |MAGAZINE PROGRAM                      |57622         |
|PSA  

In [80]:
result = (
    full_log.groupby('LogIdentifierID')
    .agg(
        F.sum(
            F.when(
                F.trim(F.col('ProgramClassCD')).isin(
                    ["COM", "PRC", "PGI", "PRO", "LOC", "SPO", "MER", "SOL"]
                ),
                F.col('duration_seconds')
            ).otherwise(0)
        ).alias('duration_commercial'),
        F.sum('duration_seconds').alias('duration_total')
    )
    .withColumn(
        "commercial_ratio", 
        F.col("duration_commercial") / F.col("duration_total")
    )
)


result.orderBy("commercial_ratio", ascending=False).show(1000, False)

+---------------+-------------------+--------------+---------------------+
|LogIdentifierID|duration_commercial|duration_total|commercial_ratio     |
+---------------+-------------------+--------------+---------------------+
|CIMT           |775                |775           |1.0                  |
|MSET           |2700               |2700          |1.0                  |
|TLNSP          |15480              |15480         |1.0                  |
|TELENO         |17790              |17790         |1.0                  |
|HPITV          |13                 |13            |1.0                  |
|TANG           |8125               |8125          |1.0                  |
|MMAX           |23333              |23582         |0.9894410991434145   |
|MPLU           |20587              |20912         |0.9844586840091814   |
|INVST          |20094              |20470         |0.9816316560820714   |
|ZT�L�          |21542              |21965         |0.9807420896881403   |
|RAPT           |17916   

#### **Handling Null Values**

In [82]:
result.count()

324

In [83]:
result_no_null = result.dropna(subset= ['commercial_ratio'])

result_no_null.orderBy(
    'commercial_ratio', ascending=False
).show(1000, False)

+---------------+-------------------+--------------+---------------------+
|LogIdentifierID|duration_commercial|duration_total|commercial_ratio     |
+---------------+-------------------+--------------+---------------------+
|CIMT           |775                |775           |1.0                  |
|MSET           |2700               |2700          |1.0                  |
|TLNSP          |15480              |15480         |1.0                  |
|TELENO         |17790              |17790         |1.0                  |
|HPITV          |13                 |13            |1.0                  |
|TANG           |8125               |8125          |1.0                  |
|MMAX           |23333              |23582         |0.9894410991434145   |
|MPLU           |20587              |20912         |0.9844586840091814   |
|INVST          |20094              |20470         |0.9816316560820714   |
|ZT�L�          |21542              |21965         |0.9807420896881403   |
|RAPT           |17916   

In [84]:
result_no_null.count()

322

In [85]:
result_no_null = result.fillna(0)

result_no_null.orderBy(
    "commercial_ratio", ascending=False
).show(1000, False)

+---------------+-------------------+--------------+---------------------+
|LogIdentifierID|duration_commercial|duration_total|commercial_ratio     |
+---------------+-------------------+--------------+---------------------+
|CIMT           |775                |775           |1.0                  |
|MSET           |2700               |2700          |1.0                  |
|TLNSP          |15480              |15480         |1.0                  |
|TELENO         |17790              |17790         |1.0                  |
|HPITV          |13                 |13            |1.0                  |
|TANG           |8125               |8125          |1.0                  |
|MMAX           |23333              |23582         |0.9894410991434145   |
|MPLU           |20587              |20912         |0.9844586840091814   |
|INVST          |20094              |20470         |0.9816316560820714   |
|ZT�L�          |21542              |21965         |0.9807420896881403   |
|RAPT           |17916   

In [86]:
result_no_null.count()

324